# 06 - The one page to keep  *(~3 minutes)*

> **Presenter script.** "That's the whole pipeline. Here is the page I actually
> want you to screenshot."

In [ ]:
# --- boilerplate: make `import minigpt` work no matter where Jupyter started ---
import pathlib
import sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "minigpt").is_dir())
sys.path.insert(0, str(ROOT))

import torch

torch.set_num_threads(4)  # plenty for a model this small; more threads is not faster
print("repo root:", ROOT)

## What we did, in one picture

```
raw messy text
   |  clean it, dedupe it, split off a validation set        (notebook 01)
   v
train / validation splits
   |  next-token prediction, ~1200 steps, ~165k parameters   (notebook 02)
   v
BASE MODEL  - writes fluent text in the style of the corpus
   |  keep training on a new domain (carefully!)             (notebook 04)
   v
DOMAIN MODEL - writes fluent text in the NEW style
   |  supervised fine-tuning on prompt/response pairs        (notebook 05)
   v
INSTRUCTION MODEL - answers the question and then stops
   |  preference tuning (DPO / RLHF)                         (concept only)
   v
ASSISTANT
```

## "I see this -> it means this -> I do this"

### Reading the loss curves

| I see | It means | I do |
|---|---|---|
| Both curves high, both still falling | **Underfitting** - not finished learning | Train longer, raise the learning rate, or grow the model |
| Both curves falling together, small gap | **Generalising** - it's working | Nothing. Keep going until validation stalls |
| Train falls, validation flattens then rises | **Overfitting** - it's memorising | More data, early stopping, dropout / weight decay, or a smaller model |
| Loss spikes, oscillates, or hits `nan` | **Diverging** - steps too big | Cut the learning rate; add warmup and gradient clipping |
| Loss sitting flat at `ln(vocab_size)` | **Not learning at all** | Check the data and labels; check the optimizer is actually stepping |
| Validation loss *below* training loss | Usually **dropout** (off at eval time) | Fine if small. If validation is much better, hunt for duplicates across the split |
| Validation loss on an **old** task got worse after new training | **Catastrophic forgetting** | Lower the learning rate; replay 10-40% of the original data |
| Loss is great but the answers are bad | You are past the point where **loss is the scoreboard** | Read the outputs; build a fixed prompt set; consider preference tuning |
| Model recites training text verbatim | **Memorisation**, usually from duplicates | Deduplicate harder (exact *and* near); add more data |

### Choosing the knobs

| Knob | Turn it up when | Turn it down when |
|---|---|---|
| **learning rate** | loss falls too slowly | loss is spiky, or you are fine-tuning an existing model |
| **steps / epochs** | still underfitting | validation loss has started rising |
| **batch size** | the loss curve is very noisy | you are short on memory |
| **model size** | it underfits on plenty of data | it overfits, or you cannot afford the compute |
| **dropout / weight decay** | it is overfitting | it is underfitting |
| **dataset size** | *always* - this is the highest-value knob there is | never |

## The five things worth remembering

1. **Data quality beats everything.** Deduplication is not housekeeping; it is
   the difference between a model that writes and a model that recites.
2. **Split off a validation set before you train.** Not after. Not "later".
3. **Two lines on a chart tell you almost everything** - and the useful one is
   the red one.
4. **Every training stage after the first is a nudge, not a rebuild.** Smaller
   learning rates, and keep some of the old data around.
5. **Once you are fine-tuning, read the outputs.** The loss stops being able to
   tell you what you need to know.

## Your model, one more time

Everything below runs on the ~165,000-parameter model we trained in about two
minutes on a CPU with no graphics card.

In [ ]:
from minigpt import data
from minigpt import train as T

tokenizer = data.load_tokenizer()
base_model, _, _ = T.load_checkpoint("base")
sft_model, _, _ = T.load_checkpoint("sft")

print(f"model size: {base_model.num_params():,} parameters\n")
T.show_sample(base_model, tokenizer, "one day ", 200, label="THE BASE MODEL (notebook 02)")

In [ ]:
print("THE FINE-TUNED MODEL (notebook 05)\n")
for q in ["what goes in the pan first?", "who followed mila home?",
          "when do i add the sage?", "how many does it serve?"]:
    print(f"Q: {q}")
    print(f"A: {T.ask(sft_model, tokenizer, q, seed=5)}")
    print()

## Where to go next

**Do the same thing, one size up**
* [nanoGPT](https://github.com/karpathy/nanoGPT) - the direct big brother of this
  repo. Same ideas, real corpora, GPU-friendly.
* [Let's build GPT (Karpathy, video)](https://www.youtube.com/watch?v=kCc8FmEb1nY)
  - builds the model in this repo from scratch, line by line.

**Use real tokenizers and real models**
* [Hugging Face `transformers`](https://huggingface.co/docs/transformers) and
  the [NLP course](https://huggingface.co/learn/nlp-course) - swap our character
  tokenizer for a real BPE one and fine-tune a small pre-trained model.
* [`trl`](https://huggingface.co/docs/trl) - SFT and DPO, implemented properly.

**Understand the data side**
* [FineWeb](https://huggingface.co/datasets/HuggingFaceFW/fineweb) - read their
  write-up on filtering and deduplication at web scale. It is notebook 01 with
  fifteen trillion tokens.

**Things to try in this repo tonight**
1. Set `USE_PREBAKED = False` everywhere and confirm the numbers reproduce.
2. In notebook 02, make the model bigger (`n_layer=6`, `n_embd=128`) and watch
   how the curves change.
3. In notebook 03, train the healthy run for 5000 steps. Does it eventually
   overfit? Where?
4. Swap the corpus: `data.load_base_corpus("shakespeare")`.
5. Write your own question/answer pairs in `minigpt/data.py` and re-run
   notebook 05.

Thanks for watching.